# WakeStudio · Train a custom openWakeWord wake word (Google Colab)

This is the **module-owned training notebook** for the
[`kws-openwakeword`](https://github.com/awareride/wake-studio/tree/main/packages/modules/kws/openwakeword)
module. Run it top-to-bottom in your own Google Colab session (free tier is
enough for the default settings) to train a custom wake-word model from a
phrase, then **download the result bundle** and import it back into
WakeStudio for in-browser testing and export.

## How it works

1. **Step 0** — set the wake phrase and training parameters (one cell).
2. **Step 1** — installs the pinned upstream [`openWakeWord`](https://github.com/dscripka/openWakeWord)
   training environment and the [`piper-sample-generator`](https://github.com/rhasspy/piper-sample-generator)
   TTS dependencies (Linux only — Colab is Linux ✓).
3. **Step 2** — downloads the same public training data the upstream
   `automatic_model_training` notebook uses (MIT room impulse responses, a
   slice of AudioSet, the FMA sample, precomputed ACAV100M openWakeWord
   features, and the validation feature set).
4. **Step 3** — writes the training **YAML config** from Step 0 parameters.
5. **Step 4** — runs the **upstream `train.py` unchanged** (bytes-identical,
   never rewritten — WakeStudio adapts to the script, per
   `docs/modules/training.md` §4): `--generate_clips`, `--augment_clips`,
   `--train_model`. The upstream script converts the trained model to ONNX
   (and TFLite when possible) into `my_custom_model/`.
6. **Step 5** — WakeStudio's `standardize-results` step lays the trained model
   and metadata down in the **standard artifact bundle**.
7. **Step 6** — download the bundle (`.zip`) and use **"Import Colab results"**
   in the WakeStudio app.

## Expected runtime

With the default parameters (1000 train + 1000 validation samples, 10 000
steps) the full run is roughly **1 hour on a free Colab T4 GPU** — mirroring
the upstream example notebook. Increase `N_SAMPLES` / `STEPS` for a stronger
model (the bundled openWakeWord release models are trained on 100 000+
samples).

## Optional keys (Settings panel)

This notebook needs **no WakeStudio credential** and no Google API key for the
default flow. If a future data source needs one (a public TTS endpoint token,
a Google API key for Drive import, …), it is read from the environment
variable `WAKE_STUDIO_*` / `*_TOKEN` (set them in the **WakeStudio Settings →
Security** section, or paste them into Step 0). **Never hard-code a secret in
this notebook** — it is committed to the repository.

## Licensing note

The trained classifier is trained from text-to-speech-generated audio and
precomputed openWakeWord features, so the resulting **model is user-owned /
commercially clean** (`provenance.json` declares `license: user-owned`). The
pre-trained openWakeWord models (CC BY-NC-SA, demo-only) are **not** bundled
into the result. Verify the licenses of any background/dataset sources you
swap in before commercial deployment.


## Step 0 · Parameters

Edit the first cell below (or leave the defaults). Every value can also be
overridden by an environment variable, so WakeStudio can pass job params
(keyboard: `wakePhrase`, `epochs`, `target`, `augment`, `quantize` from the
training panel are mapped here).


In [ ]:
# --- WakeStudio job params (from the training panel / env) -----------------
import os, uuid, time

# The wake phrase to train. Overridable via env WAKE_PHRASE.
WAKE_PHRASE = os.environ.get("WAKE_PHRASE", "hey studio")

# App-class ONNX model (target "app-class"; MCU/TFLite-Micro is a separate
# micro-wake-word notebook later). Overridable via env WAKE_TARGET.
WAKE_TARGET = os.environ.get("WAKE_TARGET", "app-class")

# Synthetic sample counts (upstream example defaults; raise for strength).
N_SAMPLES    = int(os.environ.get("WAKE_N_SAMPLES", "1000"))
N_SAMPLES_VAL= int(os.environ.get("WAKE_N_SAMPLES_VAL", "1000"))

# Training steps (upstream example default; full models often use 50 000+).
STEPS        = int(os.environ.get("WAKE_STEPS", "10000"))

# Audio augmentation toggle (background mixing + room impulse responses).
AUGMENT      = os.environ.get("WAKE_AUGMENT", "true").lower() in ("1", "true", "yes")

# Quantize export: upstream train.py already emits .tflite when possible.
QUANTIZE     = os.environ.get("WAKE_QUANTIZE", "true").lower() in ("1", "true", "yes")

# Optional keys (Settings -> Security). Read from env; never hard-coded.
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")
TTS_ENDPOINT_TOKEN = os.environ.get("TTS_ENDPOINT_TOKEN", "")

# WakeStudio job metadata
JOB_ID = os.environ.get("WAKE_JOB_ID", f"kws-openwakeword-{int(time.time()*1000)}")
MODULE_ID = "kws-openwakeword"
BACKEND = "colab"
PROVIDER = "colab"

print("wakePhrase :", WAKE_PHRASE)
print("target     :", WAKE_TARGET)
print("n_samples  :", N_SAMPLES, "| n_samples_val:", N_SAMPLES_VAL)
print("steps      :", STEPS, "| augment:", AUGMENT, "| quantize:", QUANTIZE)
print("jobId      :", JOB_ID)


## Step 1 · Environment setup

Installs the upstream `openWakeWord` package (pinned ref) and the Piper TTS
sample-generator required for synthetic data. This matches the upstream
`automatic_model_training` notebook; only the openWakeWord clone is pinned to
a fixed commit for reproducibility.


In [ ]:
# --- Environment setup (upstream automatic_model_training.ipynb) ----------
# Piper TTS sample generator (currently Linux-only; Colab is Linux).
!git clone --quiet https://github.com/rhasspy/piper-sample-generator
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install -q piper-phonemize webrtcvad

# openWakeWord (full install to support training) — pinned ref for reproducibility.
OPENWAKEWORD_REF = "7607f959"  # C-3: pinned upstream ref (latest indexed main)
!git clone --quiet https://github.com/dscripka/openwakeword
!cd openwakeword && git checkout --quiet {OPENWAKEWORD_REF}
!pip install -q -e ./openwakeword

# Other training dependencies (upstream notebook).
!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 \
    audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6 \
    tensorflow-cpu==2.8.1 tensorflow_probability==0.16.0 onnx_tf==1.10.0 \
    pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19

# Download the frozen feature models (Colab workaround, upstream notebook).
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

print("environment ready")


## Step 2 · Download training data

Same public sources as the upstream `automatic_model_training` notebook:

1. **MIT RIRs** (`davidscripka/MIT_environmental_impulse_responses`) — room
   impulse responses for augmentation.
2. **AudioSet** (`agkphysics/AudioSet`, part `bal_train09.tar`) — background
   noise; converted to 16 kHz wav.
3. **Free Music Archive** (`rudraml/fma`, `small`) — background music.
4. **Precomputed openWakeWord features** (`davidscripka/openwakeword_features`)
   — ~2000 h of negatives (ACAV100M) + the ~11 h validation feature set used
   for the false-positive-rate estimate.


In [ ]:
# --- Download data (upstream automatic_model_training.ipynb) ---------------
import os, numpy as np, scipy, datasets
from pathlib import Path
from tqdm.auto import tqdm

# 1) MIT room impulse responses
output_dir = "./mit_rirs"
os.makedirs(output_dir, exist_ok=True)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
for row in tqdm(rir_dataset, desc="MIT RIRs"):
    name = row['audio']['path'].split('/')[-1]
    path = os.path.join(output_dir, name)
    if os.path.exists(path):
        continue
    scipy.io.wavfile.write(path, 16000, (row['audio']['array']*32767).astype(np.int16))

# 2) AudioSet background (one part; full-scale training uses much more)
if not os.path.exists("./audioset/bal_train09.tar"):
    os.makedirs("./audioset", exist_ok=True)
    !wget -q -O audioset/bal_train09.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
    !cd audioset && tar -xf bal_train09.tar

output_dir = "./audioset_16k"
os.makedirs(output_dir, exist_ok=True)
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset, desc="AudioSet -> 16k"):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    path = os.path.join(output_dir, name)
    if os.path.exists(path):
        continue
    scipy.io.wavfile.write(path, 16000, (row['audio']['array']*32767).astype(np.int16))

# 3) Free Music Archive sample (1 hour of clips)
output_dir = "./fma"
os.makedirs(output_dir, exist_ok=True)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))
n_hours = 1
for i in tqdm(range(n_hours*3600//30), desc="FMA -> 16k"):
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# 4) Precomputed openWakeWord features (negatives + validation set)
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

print("data ready")


## Step 3 · Training configuration

Loads the upstream `examples/custom_model.yml`, applies the Step 0
parameters, and writes `my_model.yaml` for `train.py`.


In [ ]:
# --- Write training config (upstream train.py YAML) -----------------------
import yaml

config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)

config["target_phrase"] = [WAKE_PHRASE]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = N_SAMPLES
config["n_samples_val"] = N_SAMPLES_VAL
config["steps"] = STEPS
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25
config["background_paths"] = ['./audioset_16k', './fma']
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)

print("config written for phrase:", WAKE_PHRASE)


## Step 4 · Train the model

Runs the **upstream `train.py` unchanged** in three stages (the same flags as
the upstream notebook). stdout is teed to `train_log.txt` so Step 5 can write
`metrics.json`. If a stage fails (e.g. a transient generation error), simply
re-run that stage — the script continues until the config targets are met.


In [ ]:
# --- Stage 1: generate synthetic clips (upstream train.py) ----------------
import sys
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips 2>&1 | tee -a train_log.txt


In [ ]:
# --- Stage 2: augment the generated clips ----------------------------------
import sys
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips 2>&1 | tee -a train_log.txt


In [ ]:
# --- Stage 3: train the model (upstream train.py) -------------------------
# Saves <model_name>.onnx + .tflite into my_custom_model/ when done.
import sys
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model 2>&1 | tee -a train_log.txt


In [ ]:
# --- (Optional) Colab tflite fix (upstream notebook) -----------------------
# On Google Colab the .tflite export can fail silently; retry it manually.
def convert_onnx_to_tflite(onnx_model_path, output_path):
    import onnx, logging, tempfile
    from onnx_tf.backend import prepare
    import tensorflow as tf
    onnx_model = onnx.load(onnx_model_path)
    tf_rep = prepare(onnx_model, device="CPU")
    with tempfile.TemporaryDirectory() as tmp_dir:
        tf_rep.export_graph(os.path.join(tmp_dir, "tf_model"))
        converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, "tf_model"))
        tflite_model = converter.convert()
        with open(output_path, 'wb') as f:
            f.write(tflite_model)

model_dir = config["output_dir"]
onnx_path = f"{model_dir}/{config['model_name']}.onnx"
tflite_path = f"{model_dir}/{config['model_name']}.tflite"
if QUANTIZE and os.path.exists(onnx_path) and not os.path.exists(tflite_path):
    convert_onnx_to_tflite(onnx_path, tflite_path)
    print("tflite written:", tflite_path)


## Step 5 · Normalize into the WakeStudio standard bundle

Lays the trained model + metadata down in the standard artifact bundle
(`docs/modules/training.md` §6) and zips it for download:

```
wake-studio-results/<job-id>/
  model.onnx        (model.tflite when available)
  metrics.json      (best-effort FAR/FRR + run info parsed from the log)
  metadata.json     (jobId, moduleId, backend=colab, provider, params, trainedAtMs)
  provenance.json   (license: user-owned — commercially clean)
  config.json       (AFE/KWS/Few-Shot config snapshot used for training)
  wake-studio-results.zip (the importable bundle)
```

The zip is the **only** retrieval contract the PWA's importer
(`packages/modules/training/core/manifest.ts`) needs.


In [ ]:
# --- Build the standard artifact bundle ------------------------------------
import json, re, shutil, zipfile, uuid

model_dir = config["output_dir"]
model_name = config["model_name"]
onnx_path = f"{model_dir}/{model_name}.onnx"
tflite_path = f"{model_dir}/{model_name}.tflite"

assert os.path.exists(onnx_path), f"model not found: {onnx_path} — did Stage 3 finish?"

bundle_dir = f"wake-studio-results/{JOB_ID}"
os.makedirs(bundle_dir, exist_ok=True)

# model(s)
shutil.copy(onnx_path, os.path.join(bundle_dir, "model.onnx"))
if QUANTIZE and os.path.exists(tflite_path):
    shutil.copy(tflite_path, os.path.join(bundle_dir, "model.tflite"))

# metrics.json — best-effort parse of the training log
metrics = {"status": "ok", "note": "parsed best-effort from train_log.txt"}
if os.path.exists("train_log.txt"):
    log_text = open("train_log.txt", encoding="utf-8", errors="ignore").read()
    metrics["log_tail"] = log_text.strip().splitlines()[-20:]
    for key, pat in {
        "recall": r"recall[^0-9]*([0-9.]+)",
        "accuracy": r"accuracy[^0-9]*([0-9.]+)",
        "false_positives_per_hour": r"false[- ]?positives?[^0-9]*([0-9.]+)",
    }.items():
        m = re.search(pat, log_text, re.IGNORECASE)
        if m:
            try:
                metrics[key] = float(m.group(1))
            except ValueError:
                pass
metrics["steps"] = STEPS
metrics["epochs"] = STEPS  # upstream trains for `steps` optimizer steps
with open(os.path.join(bundle_dir, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

# metadata.json
metadata = {
    "jobId": JOB_ID,
    "moduleId": MODULE_ID,
    "backend": BACKEND,
    "provider": PROVIDER,
    "params": {
        "wakePhrase": WAKE_PHRASE,
        "target": WAKE_TARGET,
        "epochs": str(STEPS),
        "augment": str(AUGMENT).lower(),
        "quantize": str(QUANTIZE).lower(),
    },
    "trainedAtMs": int(time.time() * 1000),
}
with open(os.path.join(bundle_dir, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

# provenance.json — user-owned / commercially clean (Phase 4 license gate)
provenance = {
    "license": "user-owned",
    "sourceData": [
        {"name": "piper-sample-generator synthetic speech", "license": "MIT (code); Piper voice model license", "source": "https://github.com/rhasspy/piper-sample-generator"},
        {"name": "openWakeWord feature extractors (frozen)", "license": "Apache-2.0", "source": "https://github.com/dscripka/openWakeWord"},
        {"name": "background audio (AudioSet / FMA samples)", "license": "research-use; verify before commercial deployment", "source": "https://huggingface.co/datasets/agkphysics/AudioSet, https://huggingface.co/datasets/rudraml/fma"},
    ],
    "notes": "Trained from synthetic TTS audio + precomputed openWakeWord features. The classifier is user-owned; pre-trained openWakeWord models (CC BY-NC-SA) are NOT bundled.",
}
with open(os.path.join(bundle_dir, "provenance.json"), "w") as f:
    json.dump(provenance, f, indent=2)

# config.json — AFE/KWS/Few-Shot config snapshot used for training
config_snapshot = {
    "wakePhrase": WAKE_PHRASE,
    "target": WAKE_TARGET,
    "backend": BACKEND,
    "provider": PROVIDER,
    "model_type": config.get("model_type", "dnn"),
    "layer_size": config.get("layer_size", 32),
    "steps": STEPS,
    "n_samples": N_SAMPLES,
    "n_samples_val": N_SAMPLES_VAL,
    "augment": AUGMENT,
    "quantize": QUANTIZE,
    "clip_size_seconds": 3,
}
with open(os.path.join(bundle_dir, "config.json"), "w") as f:
    json.dump(config_snapshot, f, indent=2)

# zip it (the PWA import contract)
zip_path = f"{bundle_dir}/wake-studio-results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in ("model.onnx", "model.tflite", "metrics.json", "metadata.json", "provenance.json", "config.json"):
        p = os.path.join(bundle_dir, name)
        if os.path.exists(p):
            zf.write(p, arcname=f"{JOB_ID}/{name}")

print("bundle ready:")
for name in sorted(os.listdir(bundle_dir)):
    print("  ", name, os.path.getsize(os.path.join(bundle_dir, name)), "bytes")
print("download:", zip_path)


## Step 6 · Import the bundle back into WakeStudio

1. Download **`wake-studio-results/<job-id>/wake-studio-results.zip`**
   (left-hand file browser in Colab → right-click → Download).
2. In the WakeStudio app open the **Training** panel for `kws-openwakeword`
   and choose **Import Colab results**; pick the zip.
3. The importer validates `metadata.json` + `provenance.json` (the manifest is
   shared by every backend) and registers the model for **in-browser testing**
   and **export** (the export license gate reads `provenance.json` — this
   model is `user-owned`, so it is exportable).

No WakeStudio server is involved at any point — your Google account is the
only credential.
